# 🎓 Masterclass en Data Science
## Ejercicio Práctico — Notebook del Estudiante
### Predicción de Churn Bancario con Machine Learning

---
**Dataset:** Sintético (banco retail europeo, ~10.000 clientes)  
**Objetivo:** Clasificación binaria — predecir clientes con riesgo de churn  

> **📌 INSTRUCCIONES:** Completa las celdas marcadas con `# TODO`. Las celdas de generación del dataset y limpieza básica ya están resueltas — comienza a programar a partir de la Parte 2.

In [ ]:
# ── IMPORTS ────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (roc_auc_score, f1_score, recall_score, precision_score,
                             roc_curve, precision_recall_curve, classification_report,
                             ConfusionMatrixDisplay, average_precision_score)
import xgboost as xgb
import shap

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'
PALETTE = {'churn': '#D35400', 'no_churn': '#1A3A6B'}
print('✅ Imports correctos')

---
## PARTE 1 — Generación del dataset y calidad del dato
*(En la práctica real, aquí cargarías el CSV con `pd.read_csv`)*

**Esta parte está resuelta. Ejecútala tal cual y observa la estructura del dataset.**

In [ ]:
# ── GENERACIÓN DEL DATASET SINTÉTICO ──────────────────────────────────────────
np.random.seed(42)
N = 10_000

credit_score    = np.random.randint(300, 851, N).astype(float)
country_raw     = np.random.choice(['France','Germany','Spain'], N, p=[0.50, 0.25, 0.25])
gender          = np.random.choice(['Male','Female'], N, p=[0.545, 0.455])
age             = np.random.normal(38.9, 10.5, N).clip(18, 92).round().astype(float)
tenure          = np.random.randint(0, 11, N).astype(float)
balance         = np.where(np.random.rand(N) < 0.29, 0,
                  np.random.normal(76_485, 62_397, N).clip(0, 250_000))
num_products    = np.random.choice([1,2,3,4], N, p=[0.50, 0.46, 0.025, 0.015])
has_credit_card = np.random.choice([0,1], N, p=[0.29, 0.71])
is_active       = np.random.choice([0,1], N, p=[0.485, 0.515])
salary          = np.random.uniform(11_580, 199_992, N)

logit = (-2.8
         + 0.03 * (age - 38)
         + 0.6  * (country_raw == 'Germany').astype(float)
         - 0.5  * is_active
         + np.where(num_products >= 3, 1.2, 0)
         + 0.4  * (balance == 0).astype(float)
         - 0.3  * (gender == 'Female').astype(float)
         + np.random.normal(0, 0.5, N))
prob_churn = 1 / (1 + np.exp(-logit))
churned = (np.random.rand(N) < prob_churn).astype(int)

df = pd.DataFrame({
    'customer_id'     : 15_000_001 + np.arange(N),
    'credit_score'    : credit_score,
    'country'         : country_raw,
    'gender'          : gender,
    'age'             : age,
    'tenure'          : tenure,
    'balance'         : balance.round(2),
    'num_products'    : num_products,
    'has_credit_card' : has_credit_card,
    'is_active_member': is_active,
    'estimated_salary': salary.round(2),
    'churned'         : churned,
})

# Errores intencionados
null_cs  = np.random.choice(N, int(N*0.05), replace=False)
null_age = np.random.choice(N, int(N*0.03), replace=False)
df.loc[null_cs,  'credit_score'] = np.nan
df.loc[null_age, 'age']          = np.nan

dup_idx = np.random.choice(N, 50, replace=False)
df = pd.concat([df, df.iloc[dup_idx]], ignore_index=True)

df.loc[np.random.choice(df.index, 20), 'age'] = np.random.choice([150, 200, -5], 20)

mask = np.random.choice(df.index, 200, replace=False)
df.loc[mask, 'country'] = df.loc[mask, 'country'].str.lower()
mask2 = np.random.choice(df.index, 100, replace=False)
df.loc[mask2, 'country'] = df.loc[mask2, 'country'].str.upper()

df.loc[np.random.choice(df.index, 30), 'balance'] = np.random.uniform(-5000, -1, 30).round(2)

print(f'Dataset creado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'Churn rate base: {churned.mean():.1%}')
df.head()

### 1.1 Inspección inicial

Ejecuta las siguientes celdas y observa los problemas de calidad que aparecen.

In [ ]:
print('── SHAPE ──────────────────────────────────────────')
print(df.shape)

print('\n── DTYPES + NULL COUNT ────────────────────────────')
df.info()

print('\n── ESTADÍSTICOS ───────────────────────────────────')
df.describe(include='all').round(2)

In [ ]:
# ── REPORTE DE CALIDAD ────────────────────────────────────────────────────────
print('═'*55)
print('REPORTE DE CALIDAD DEL DATASET')
print('═'*55)

nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(2)
print('\n📋 VALORES NULOS:')
print(pd.DataFrame({'count': nulls[nulls>0], '%': null_pct[null_pct>0]}))

n_dup = df.duplicated(subset=[c for c in df.columns if c != 'customer_id']).sum()
print(f'\n🔑 DUPLICADOS (excl. customer_id): {n_dup}')

print('\n🌍 VARIANTES DE country:')
print(df['country'].value_counts())

print(f"\n⚠️  age fuera de rango [18, 100]: {((df['age'] < 18) | (df['age'] > 100)).sum()}")
print(f"⚠️  balance negativo: {(df['balance'] < 0).sum()}")

### 1.2 Limpieza del dataset

Esta parte también está resuelta — obsérvala con atención porque establece el `df_clean` que usarás en los pasos siguientes.

In [ ]:
df_clean = df.copy()

# 1. Normalizar country → Title case
df_clean['country'] = df_clean['country'].str.strip().str.title()
print('✅ country normalizado:', df_clean['country'].unique())

# 2. Eliminar duplicados
n_before = len(df_clean)
df_clean.drop_duplicates(
    subset=[c for c in df_clean.columns if c != 'customer_id'],
    keep='first', inplace=True
)
print(f'✅ Duplicados eliminados: {n_before - len(df_clean)}')

# 3. Valores imposibles en age → NaN (se imputan en el Pipeline)
impossible_age = (df_clean['age'] < 18) | (df_clean['age'] > 100)
df_clean.loc[impossible_age, 'age'] = np.nan
print(f'✅ age imposibles → NaN: {impossible_age.sum()}')

# 4. Balances negativos → NaN
neg_bal = df_clean['balance'] < 0
df_clean.loc[neg_bal, 'balance'] = np.nan
print(f'✅ balance negativos → NaN: {neg_bal.sum()}')

# 5. Eliminar customer_id (identificador, no predictivo)
df_clean.drop(columns=['customer_id'], inplace=True)

print(f'\n📐 Dataset limpio: {df_clean.shape[0]:,} filas × {df_clean.shape[1]} columnas')
print(f'Churn rate: {df_clean["churned"].mean():.1%}')

---
## PARTE 2 — EDA Avanzado

Analiza la variable objetivo y las principales variables predictoras.

### 2.1 Distribución del target

In [ ]:
# TODO: Visualiza la distribución de la variable 'churned'.
# Crea una figura con 2 subplots lado a lado:
#   - Izquierda: gráfico de barras con el conteo absoluto de cada clase (0 / 1)
#   - Derecha:   gráfico de tarta (pie) con la proporción
# Usa los colores PALETTE['no_churn'] y PALETTE['churn'] para las clases.
# Añade anotaciones numéricas sobre las barras.
# Al final, imprime un aviso sobre el desbalance del dataset.

# Tu código aquí:


### 2.2 Distribuciones numéricas por clase

In [ ]:
# TODO: Crea boxplots de las variables numéricas separando por clase de churn.
# Variables a representar: 'credit_score', 'age', 'tenure', 'balance', 'estimated_salary'
# Pista: puedes usar df_clean.boxplot(column=col, by='churned', ax=ax)
# ¿Qué variable muestra la diferencia más clara entre churners y no-churners?

# Tu código aquí:


### 2.3 Churn rate por segmento

In [ ]:
# TODO: Calcula y visualiza el churn rate para 3 segmentos distintos:
#
# a) Por 'country'  →  ¿qué país tiene más churn?
#    Pista: df_clean.groupby('country')['churned'].mean()
#
# b) Por 'num_products'  →  ¿a partir de cuántos productos el churn dispara?
#
# c) Por rango de edad  →  usa pd.cut con bins=[17,30,45,60,100]
#    Etiquetas: ['18-30','31-45','46-60','61+']
#    (Recuerda eliminar la columna auxiliar 'age_band' al final)
#
# Representa los 3 en una figura con 3 subplots.
# Destaca con PALETTE['churn'] el segmento de mayor churn.

# Tu código aquí:


### 2.4 Matriz de correlación

In [ ]:
# TODO: Genera un heatmap de correlaciones (Pearson) entre las variables numéricas.
# Incluye la variable 'churned' en el análisis.
# Usa la máscara triangular superior para evitar duplicados:
#   mask = np.triu(np.ones_like(corr, dtype=bool))
# Imprime las 5 variables con mayor correlación (en valor absoluto) con 'churned'.

# Tu código aquí:


### 💡 Tus insights del EDA

Anota aquí (como texto Markdown) al menos **3 hallazgos relevantes** del EDA:

1. *Escribe tu hallazgo aquí...*
2. *Escribe tu hallazgo aquí...*
3. *Escribe tu hallazgo aquí...*

---
## PARTE 3 — Feature Engineering

Crea nuevas variables a partir de las existentes. Las mejores features combinan variables que el EDA ha identificado como relevantes.

**Objetivo:** crear al menos 4 features nuevas. Las primeras 2 están guiadas, las otras 3 son libres.

| Feature | Descripción | Rationale |
|---|---|---|
| `balance_per_product` | `balance / num_products` | Clientes con muchos productos y poco balance tienen más riesgo |
| `is_zero_balance` | `1` si `balance == 0` | Las cuentas sin saldo están posiblemente dormidas |
| `wealth_score` | `balance + estimated_salary` | Proxy del valor patrimonial del cliente |
| `is_senior` | `1` si `age >= 46` | El EDA muestra mayor churn en clientes de mediana edad |
| `inactive_zero_bal` | `1` si inactivo Y balance=0 | Doble señal de abandono |


In [ ]:
df_fe = df_clean.copy()

# Feature 1: Balance por producto (ya guiada)
df_fe['balance_per_product'] = df_fe['balance'] / df_fe['num_products']

# Feature 2: Saldo cero (ya guiada)
df_fe['is_zero_balance'] = (df_fe['balance'] == 0).astype(int)

# TODO: Crea las features 3, 4 y 5 de la tabla de arriba (o las tuyas propias):

# Feature 3: wealth_score
# df_fe['wealth_score'] = ...

# Feature 4: is_senior  (usa .astype(float) para compatibilidad con valores nulos)
# df_fe['is_senior'] = ...

# Feature 5: inactive_zero_bal
# df_fe['inactive_zero_bal'] = ...

print('Nuevas features creadas:')
new_feats = ['balance_per_product','is_zero_balance','wealth_score','is_senior','inactive_zero_bal']
print(df_fe[new_feats].describe().round(2))

In [ ]:
# ── SPLIT TRAIN / TEST ────────────────────────────────────────────────────────
# Esta celda está resuelta — ejecútala para tener X_train, X_test, y_train, y_test

TARGET = 'churned'
X = df_fe.drop(columns=[TARGET])
y = df_fe[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]:,} filas | Test: {X_test.shape[0]:,} filas')
print(f'Churn en train: {y_train.mean():.1%} | en test: {y_test.mean():.1%}')

---
## PARTE 4 — Modelado y Validación

Construirás un **Pipeline de sklearn** con:
1. Preprocesamiento diferenciado por tipo de columna (`ColumnTransformer`)
2. Clasificador `XGBClassifier`
3. Validación con `StratifiedKFold(n_splits=5)`

### 4.1 Construir el Pipeline

In [ ]:
# ── DEFINIR COLUMNAS POR TIPO ─────────────────────────────────────────────────
cat_cols = ['country', 'gender']
num_cols = ['credit_score','age','tenure','balance','num_products','has_credit_card',
            'is_active_member','estimated_salary','balance_per_product',
            'wealth_score','is_senior','inactive_zero_bal']
bin_cols = ['is_zero_balance']  # ya es 0/1, no necesita scaling

# TODO: Define el preprocesador para columnas numéricas:
#   - Imputar con la mediana (SimpleImputer strategy='median')
#   - Escalar con StandardScaler

# num_transformer = Pipeline(steps=[...])

# TODO: Define el preprocesador para columnas categóricas:
#   - Imputar con la moda (strategy='most_frequent')
#   - OneHotEncoder con drop='first' y handle_unknown='ignore'
#     Añade sparse_output=False para obtener un array denso

# cat_transformer = Pipeline(steps=[...])

# Ya resuelta: transformer para binarias (solo imputar, no escalar)
bin_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
])

# TODO: Ensambla el ColumnTransformer con los tres transformers:
#   - ('num', num_transformer, num_cols)
#   - ('cat', cat_transformer, cat_cols)
#   - ('bin', bin_transformer, bin_cols)

# preprocessor = ColumnTransformer(transformers=[...])

# TODO: Crea el XGBClassifier. Parámetros clave:
#   - n_estimators=400, max_depth=5, learning_rate=0.05
#   - subsample=0.8, colsample_bytree=0.7
#   - scale_pos_weight = nº negativos / nº positivos  ← maneja el desbalance
#     Pista: (y_train == 0).sum() / (y_train == 1).sum()
#   - eval_metric='logloss', random_state=42, n_jobs=-1

# model_xgb = xgb.XGBClassifier(...)

# TODO: Construye el Pipeline final con dos pasos: 'prep' y 'model'

# pipeline = Pipeline(steps=[...])
# print('✅ Pipeline construido')
# print(pipeline)

### 4.2 Validación cruzada estratificada

In [ ]:
# TODO: Ejecuta una validación cruzada Stratified 5-Fold sobre el pipeline.
#
# Pasos:
# 1. Crea un StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# 2. Llama a cross_validate() con el pipeline, X_train, y_train y el cv anterior.
#    Métricas a evaluar: ['roc_auc','f1','recall','precision','average_precision']
#    Añade return_train_score=False, n_jobs=-1
# 3. Imprime la media y desviación típica de cada métrica.
#
# Pregunta de reflexión: ¿por qué usamos StratifiedKFold en lugar de KFold normal?

# Tu código aquí:


### 4.3 Evaluación en test set y elección de umbral

In [ ]:
# TODO: Entrena el pipeline sobre todo el train set y evalúa en test.
#
# 1. Llama a pipeline.fit(X_train, y_train)
# 2. Obtén las probabilidades con pipeline.predict_proba(X_test)[:, 1]
# 3. Aplica un umbral de clasificación de 0.40:
#    y_pred = (y_prob >= 0.40).astype(int)
# 4. Imprime: AUC-ROC, PR-AUC, F1, Recall, Precision y el classification_report
#
# Pista: para PR-AUC usa average_precision_score(y_test, y_prob)

THRESHOLD = 0.40

# Tu código aquí:


In [ ]:
# TODO: Dibuja las curvas ROC y Precision-Recall.
#
# Figura con 2 subplots:
# - Izquierda: curva ROC (fpr vs tpr) + diagonal aleatoria + marca el punto del umbral 0.40
#   Usa roc_curve(y_test, y_prob) para obtener fpr, tpr, thresholds
#   Marca el umbral con: idx = np.argmin(np.abs(thresholds - THRESHOLD))
# - Derecha: curva PR (recall vs precision) + baseline (churn rate medio) + punto del umbral
#   Usa precision_recall_curve(y_test, y_prob)
#
# Al final, interpreta los resultados con un print:
#   "Con umbral X: Precision=Y% (de cada 100 alertas, Y son churn real)"
#   "Recall=Z% (capturamos el Z% de todos los churners)"

# Tu código aquí:


In [ ]:
# TODO: Dibuja la matriz de confusión con ConfusionMatrixDisplay.from_predictions()
# Etiquetas: ['No Churn','Churn']
# Incluye el umbral usado en el título.

# Tu código aquí:


---
## PARTE 5 — Explicabilidad con SHAP

SHAP (SHapley Additive exPlanations) descompone la predicción de cada instancia en contribuciones por feature. Para modelos basados en árboles como XGBoost, `TreeExplainer` calcula SHAP values exactos de forma eficiente.

**Flujo:**
1. Extraer el conjunto de test ya transformado (el `TreeExplainer` recibe los datos pre-procesados)
2. Crear el `TreeExplainer` a partir del modelo dentro del pipeline
3. Calcular SHAP values y visualizarlos

In [ ]:
# ── EXTRAER NOMBRES DE FEATURES TRAS EL PREPROCESAMIENTO ─────────────────────
# Esta celda está resuelta — los nombres son necesarios para los plots

ohe_cols = pipeline.named_steps['prep'].named_transformers_['cat'].named_steps['ohe'].get_feature_names_out(cat_cols).tolist()
feature_names = num_cols + ohe_cols + bin_cols
print(f'Total features tras preprocesamiento: {len(feature_names)}')
print(feature_names)

In [ ]:
# TODO: Calcula los SHAP values para el conjunto de test.
#
# 1. Transforma X_test con el preprocesador:
#    X_test_prep = pipeline.named_steps['prep'].transform(X_test)
#    X_test_prep_df = pd.DataFrame(X_test_prep, columns=feature_names)
#
# 2. Crea el explainer:
#    explainer = shap.TreeExplainer(pipeline.named_steps['model'])
#
# 3. Calcula los shap values:
#    shap_values = explainer(X_test_prep_df)
#
# 4. Imprime la shape de shap_values.values para verificar

# Tu código aquí:


In [ ]:
# TODO: Genera el SHAP Summary Plot (beeswarm).
#
# Usa: shap.summary_plot(shap_values, X_test_prep_df, show=False, max_display=15)
# Añade un título descriptivo y ajusta el layout.
#
# Pregunta: ¿cuáles son las 3 features con mayor impacto global?
# ¿Coinciden con lo que el EDA sugería?

# Tu código aquí:


In [ ]:
# TODO: Calcula la importancia media absoluta de SHAP por feature.
#
# mean_abs_shap = pd.Series(
#     np.abs(shap_values.values).mean(axis=0),
#     index=feature_names
# ).sort_values(ascending=False)
#
# Imprime el top 10.

# Tu código aquí:


In [ ]:
# TODO: Analiza dos clientes concretos: el de mayor y el de menor probabilidad de churn.
#
# 1. Identifica sus índices:
#    high_risk_idx = np.argmax(y_prob)
#    low_risk_idx  = np.argmin(y_prob)
#
# 2. Imprime sus características principales (age, balance, num_products, is_active_member, country)
#    y sus probabilidades predichas.
#
# 3. Para cada uno, haz un gráfico de barras horizontal (barh) con las top 8 features
#    por valor absoluto de SHAP:
#    - Barras rojas (PALETTE['churn']) para SHAP > 0 (aumenta churn)
#    - Barras azules (PALETTE['no_churn']) para SHAP < 0 (reduce churn)
#    - Eje X: 'SHAP value (impacto en log-odds de churn)'
#    - Línea vertical en x=0
#
# Pista para obtener el top 8:
#   sv = shap_values[high_risk_idx].values
#   top_idx = np.argsort(np.abs(sv))[-8:][::-1]

# Tu código aquí:


---
## Conclusiones y recomendaciones de negocio

Responde a las siguientes preguntas en esta celda Markdown:

**1. ¿Cuáles son los 3 principales factores de riesgo de churn según el modelo SHAP?**

*Tu respuesta...*

**2. ¿A qué segmento de clientes priorizarías para las acciones de retención?**

*Tu respuesta...*

**3. ¿Mantendrías el umbral de clasificación en 0.40? ¿Por qué?**  
*(Pista: el coste de una acción de retención es ~20€ y el valor de un cliente retenido es ~180€)*

*Tu respuesta...*

**4. ¿Qué limitaciones tiene este modelo? ¿Qué mejorarías?**

*Tu respuesta...*